# Kim & Shin (2025) on the Allen Visual Coding Neuropixels data: decoding and shuffled noise correlations

Paper: Kim, J. H. & Shin, H. (2025), bioRxiv doi:10.1101/2025.06.30.662469 (July 2025 preprint, the text the claims file was written from).

Data: the Allen Brain Observatory Visual Coding Neuropixels "Brain Observatory 1.1" sessions as deposited in dandiset 000021, version 0.251116.2246 (doi:10.48324/dandi.000021/0.251116.2246), the only published version, streamed. These claims are `data: elsewhere`. Sessions, units and spike counts are those of the second notebook (`reproduce_kim_2025_allen.ipynb`), which also holds the data checks: V1 units passing the Methods' three quality criteria and labelled "good" (an assumption, stated there), spikes in the 250 ms from each natural scenes onset, and the 30 sessions left after the Methods' exclusion of sessions with more than 10% of trials silent in all V1 units.

| Claim | Tested here |
|---|---|
| fig2b-decoding-falls-with-slope | blocked as too large; run in part, at as-is and slopes 0 and 1 over the 30 sessions, and at slope 2 over the first four included sessions, which times it. The estimate for slope 2 over 30 sessions and for the criterion's 22 conditions is computed below from these times. |
| figs4c-shuffling-alters-dependence | decoding at as-is and slope 0; RSM similarity within and across mice (Figures 4C and 4E) at every slope |

Seeds: each claim's generator is seeded with `zlib.crc32(b"kim-2025/<claim id>")`; a second generator of a claim adds 1. The decoding and the shuffled RSM comparison are run under the claim's seed only, for lack of time. The seed check of the RSM-based results of the second notebook (fig4c, fig4e, figs6), under the seed plus 1000, 2000, 3000 and 4000, is run at the end of this notebook, which loads the same data.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import time, zlib, itertools, resource
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
from scipy.stats import spearmanr

import kim2025 as K

def wilcoxon(x, y=None, **kw):
    """scipy's Wilcoxon signed-rank test, with p = 1 when every difference is zero."""
    from scipy.stats import wilcoxon as _w
    d = np.asarray(x) - (0 if y is None else np.asarray(y))
    if np.all(d == 0):
        return type("R", (), {"pvalue": 1.0, "statistic": 0.0})()
    return _w(x, y, **kw) if y is not None else _w(x, **kw)

T0 = time.time()
pd.set_option("display.width", 220)
SLOPES = [float(s) for s in K.TARGET_SLOPES]
CONDS = ["as-is"] + SLOPES
def seed(claim):
    return zlib.crc32(f"kim-2025/{claim}".encode())
paths = [p for p in K.asset_paths(K.ABO_ID, K.ABO_VERSION) if "probe" not in p]
print(f"dandiset {K.ABO_ID} version {K.ABO_VERSION}: {len(paths)} session files")

## Data check

In [ ]:
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    SQ = list(ex.map(K.abo_session, paths))
S = [K.restrict(s, s["good"]) for s in SQ]
V1 = {i: S[i]["X"][:, S[i]["area"] == "V1"] for i in range(len(S))}
sil = pd.Series({i: K.silent_fraction(V1[i]) for i in V1})
keep = [i for i in V1 if sil[i] <= K.SILENT_TRIAL_LIMIT]
print(f"loaded in {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
nv1 = pd.Series({i: V1[i].shape[1] for i in keep})
reps = {i: np.unique(np.unique(S[i]["labels"], return_counts=True)[1]) for i in keep}
print(f"excluded sessions (more than 10% of trials silent in all V1 units): {[i for i in V1 if i not in keep]}; included {len(keep)}")
print(f"V1 units (good) per included session: mean {nv1.mean():.1f}, range {nv1.min()} to {nv1.max()}, total {nv1.sum()}")
print("stimuli per session:", {len(np.unique(S[i]["labels"])) for i in keep}, "; repeats per stimulus:", {tuple(r) for r in reps.values()})
allc = np.concatenate([V1[i].ravel() for i in keep])
print(f"V1 spike counts per 250 ms: min {allc.min():.0f}, max {allc.max():.0f}, mean {allc.mean():.3f}")

## fig2b-decoding-falls-with-slope

> Linear SVM decoding accuracy of image identity decreases monotonically as the slope increases, above the chance level of 1/119.

sklearn `SVC(kernel="linear")` with default C, 10-fold `StratifiedKFold` without shuffling after a random reordering of the trials within each stimulus, counts centred by the training mean in each fold, mean accuracy per session (RSA_decoding.py, `decode_ABO`, at ac3300b; `K.decode_session`). All sessions have 50 repeats of each of the 119 stimuli. Run at as-is and slopes 0 and 1 over the 30 sessions (M2), and at slope 2 over four sessions, for the reason given in the header. The criterion (O2 with a decrease at every 0.1 step, and slopes 0 and 2 against as-is over the sessions) cannot be judged without slope 2 in every session, so the claim stays blocked; what is printed is the part that could be run: slope 0 against slope 1 and against as-is by the Wilcoxon signed-rank test (M4), slope 2 against as-is in four sessions, and accuracy against the chance level 1/119.

In [ ]:
SEED_2B, SEED_4C = seed("fig2b-decoding-falls-with-slope"), seed("figs4c-shuffling-alters-dependence")
DEC = [(None, False), (1.0, False), (None, True), (0.0, False), (0.0, True)]
jobs = [(c, sh, i) for c, sh in DEC for i in keep]          # slowest conditions first

def djob(a):
    c, sh, i = a
    t = time.time()
    acc = K.decode_session(V1[i], S[i]["labels"], c, SEED_2B, shuffle_neurons=sh, shuffle_seed=SEED_4C)
    return c, sh, i, acc, time.time() - t
t = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    dres = list(ex.map(djob, jobs))
print(f"{len(jobs)} decodings in {(time.time() - t) / 60:.1f} min")
dec = pd.DataFrame(dres, columns=["cond", "shuffled", "session", "acc", "seconds"])
dec["cond"] = dec.cond.map(lambda c: "as-is" if c is None or (isinstance(c, float) and np.isnan(c)) else float(c))
print("CPU seconds per decoding, by condition:", dec.groupby(["shuffled", "cond"]).seconds.agg(["median", "max"]).round(0).to_dict("index"))
A = dec[~dec.shuffled].pivot(index="session", columns="cond", values="acc")
Ash = dec[dec.shuffled].pivot(index="session", columns="cond", values="acc")

In [ ]:
m = A.mean()
p0, p01 = wilcoxon(A[0.0], A["as-is"]).pvalue, wilcoxon(A[0.0], A[1.0]).pvalue
print("accuracy per session (original data):")
print(A[["as-is", 0.0, 1.0]].round(3).to_string())
print(f"mean accuracy: as-is {m['as-is']:.4f}, slope 0 {m[0.0]:.4f}, slope 1 {m[1.0]:.4f} (slope 2 not run)")
print(f"slope 0 above slope 1 in {int((A[0.0] > A[1.0]).sum())} of {len(A)} sessions, Wilcoxon p = {p01:.3g}; "
      f"slope 0 against as-is p = {p0:.3g} (higher: {m[0.0] > m['as-is']})")
print(f"chance 1/119 = {1 / 119:.4f}; smallest session accuracy at any condition run {A.min().min():.4f}; "
      f"all sessions above chance at every condition run: {bool((A > 1 / 119).all().all())}")

### Slope 2 on four sessions

The original data decoded at slope 2 in the first four included sessions, four decodings in parallel on the 4 CPUs, as above. The times give the estimate for slope 2 over the 30 sessions and for the criterion's 22 conditions; the accuracies are compared with as-is in those sessions.

In [ ]:
sub4 = keep[:4]
t = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    d2 = list(ex.map(djob, [(2.0, False, i) for i in sub4]))
wall2 = time.time() - t
sec2 = np.array([r[4] for r in d2])
print(f"slope 2, sessions {sub4}: {wall2 / 60:.1f} min wall time; seconds per decoding {np.round(sec2).astype(int).tolist()} (median {np.median(sec2):.0f})")
per_cond = dec[~dec.shuffled].groupby("cond").seconds.median()
print("median seconds per decoding over 30 sessions, as-is, slope 0, slope 1:", per_cond.round(0).to_dict())
print(f"estimate, slope 2 over 30 sessions on 4 CPUs: {30 * np.median(sec2) / 4 / 60:.0f} min")
mid = np.interp(np.arange(1, 20) / 10, [0, 1, 2], [per_cond[0.0], per_cond[1.0], np.median(sec2)])
print(f"estimate, the criterion's 22 conditions (as-is, slopes 0 to 2 by 0.1) over 30 sessions on 4 CPUs, with slopes between 0, 1 and 2 "
      f"interpolated linearly in time: {30 * (per_cond['as-is'] + per_cond[0.0] + mid.sum() + np.median(sec2)) / 4 / 3600:.1f} h")
a2 = pd.Series({i: acc for _, _, i, acc, _ in d2})
print(pd.DataFrame({"as-is": A.loc[sub4, "as-is"], "slope 0": A.loc[sub4, 0.0], "slope 1": A.loc[sub4, 1.0], "slope 2": a2}).round(3).to_string())
print(f"slope 2 below as-is in {int((a2 < A.loc[sub4, 'as-is']).sum())} of {len(sub4)} sessions; all above chance 1/119: {bool((a2 > 1 / 119).all())}")

**Result: fig2b-decoding-falls-with-slope blocked, too large.** As-is and slopes 0 and 1 were run over the 30 sessions and slope 2 over four, and the criterion needs slope 2 in every session and every 0.1 step. In the part that ran, accuracy is higher at slope 0 than at slope 1 in 30 of 30 sessions (0.787 against 0.424, p = 1.9e-09) and higher at slope 0 than as-is (0.403, p = 1.9e-09), and every session is above the chance level 1/119 = 0.0084 at every condition run (lowest 0.169 over the 30 sessions, 0.132 at slope 2). In the four sessions run at slope 2, accuracy at slope 2 is below as-is in each (0.212, 0.188, 0.132, 0.134 against 0.356, 0.255, 0.185, 0.446). The median decoding at slope 2 takes more than ten times as long as at as-is, slope 0 or slope 1 (printed above), so slope 2 over the 30 sessions takes, by the estimate printed above, close to an hour on the 4 CPUs, near the sixty-minute limit of a notebook on its own, and the criterion's 22 conditions at several hours (the estimate above interpolates the time between slopes 1 and 2 linearly, which is rough). As-is accuracy in session 1 is 0.356.

## figs4c-shuffling-alters-dependence

> Destroying pairwise noise correlations by shuffling trials within stimulus for each neuron changes how decoding accuracy and RSM similarity depend on the slope.

Each neuron's trials are permuted within each stimulus before the slope change (Discussion, paragraph 5; for the decoding inside `K.decode_session`, seeded with the claim's generator, with the trial order and folds of the original decoding, set by the seed of fig2b-decoding-falls-with-slope, so that original and shuffled differ only by the shuffle; for the RSMs with `K.shuffle_within_stimulus`, seeded with the claim's second generator). The analyses of Figures 2B (at as-is and slope 0, the conditions that fit in the time), 4C and 4E are repeated on the shuffled data and compared with the original by the Wilcoxon signed-rank test at each slope run, across sessions (2B, 4C) or session pairs (4E). The RSM measures use the same seeds and procedure as the second notebook (`K.random_measures`): RSMs from a random order of trials within stimulus, and for 4C ten random half splits of the neurons. The claims file makes this claim depend on fig2b-decoding-falls-with-slope, which is blocked for the cost of its 22 conditions; the comparison here needs only the decoding at the conditions run, so it is tested.

In [ ]:
rows = []
for c in ["as-is", 0.0]:
    w = wilcoxon(A[c], Ash[c])
    rows.append(dict(cond=c, original=round(A[c].mean(), 4), shuffled=round(Ash[c].mean(), 4), p=float(f"{w.pvalue:.3g}")))
d2b = pd.DataFrame(rows).set_index("cond")
print("Figure 2B, original against shuffled (30 sessions):"); print(d2b.to_string())
print("conditions with p < 0.05:", int((d2b.p < 0.05).sum()), "of", len(d2b))

In [ ]:
SEEDS = {k: seed(k) for k in ["fig4e-rsm-similarity-across-mice", "fig4c-rsm-similarity-within-mice",
                              "fig4a-overlap-consistency-within-mice", "figs6-higher-areas-consistency"]}
rsm_sp = lambda A_, B_: spearmanr(A_.ravel(), B_.ravel(), nan_policy="omit").statistic
pairs = list(itertools.combinations(keep, 2))

def sjob(a):
    i, off, shuffled = a
    X = V1[i]
    if shuffled:
        X = K.shuffle_within_stimulus(X, S[i]["labels"], np.random.default_rng(SEED_4C + 1 + off))
    return K.random_measures(X, S[i]["labels"], {k: v + off for k, v in SEEDS.items()})

def rsm_compare(off):
    with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
        res = list(ex.map(sjob, [(i, off, sh) for sh in (False, True) for i in keep]))
    orig, shuf = dict(zip(keep, res[:len(keep)])), dict(zip(keep, res[len(keep):]))
    w4c = pd.DataFrame({i: orig[i]["rsm_half"] for i in keep}).T[CONDS]
    s4c = pd.DataFrame({i: shuf[i]["rsm_half"] for i in keep}).T[CONDS]
    w4e = pd.DataFrame({p: {c: rsm_sp(orig[p[0]]["rsm"][c], orig[p[1]]["rsm"][c]) for c in CONDS} for p in pairs}).T
    s4e = pd.DataFrame({p: {c: rsm_sp(shuf[p[0]]["rsm"][c], shuf[p[1]]["rsm"][c]) for c in CONDS} for p in pairs}).T
    p4c = pd.Series({c: wilcoxon(w4c[c], s4c[c]).pvalue for c in CONDS})
    p4e = pd.Series({c: wilcoxon(w4e[c], s4e[c]).pvalue for c in CONDS})
    return w4c, s4c, w4e, s4e, p4c, p4e

t = time.time()
w4c, s4c, w4e, s4e, p4c, p4e = rsm_compare(0)
print(f"RSM measures, original and shuffled, in {(time.time() - t) / 60:.1f} min")
for name, w_, s_, p_ in (("Figure 4C, within sessions (30 sessions)", w4c, s4c, p4c), ("Figure 4E, across sessions (435 pairs)", w4e, s4e, p4e)):
    print(name)
    print(pd.DataFrame({"original": w_.mean(), "shuffled": s_.mean(), "p": p_}).T.round(4).to_string())
    print(f"   slopes (of 21, and as-is) with p < 0.05: {int((p_[SLOPES] < 0.05).sum())}, as-is p = {p_['as-is']:.3g}; "
          f"peak slope original {w_[SLOPES].mean().idxmax()}, shuffled {s_[SLOPES].mean().idxmax()}")

**Result: figs4c-shuffling-alters-dependence reproduced.** Shuffling each neuron's trials within stimulus changes all three analyses at one or more slopes. Decoding (2B), with the same trial order and folds for the original and shuffled data: 0.435 shuffled against 0.403 original as-is (p = 5e-05) and 0.764 against 0.787 at slope 0 (p = 0.029), the only conditions run in all sessions. Within-session RSM similarity (4C): different at 16 of 21 slopes and as-is (as-is p = 3.9e-07); the shuffled curve peaks at 1.0, the original at 1.1. Across-session RSM similarity (4E): different at all 21 slopes and as-is (as-is p = 1.4e-29); the shuffled curve peaks at 1.3, the original at 1.1. Only the claim's seed was run for these comparisons.

### Seed check for the RSM-based results of the second notebook (fig4c, fig4e, figs6)

The RSM measures of `reproduce_kim_2025_allen.ipynb` repeated with every seed plus 1000, 2000, 3000 and 4000, on the same sessions and units.

In [ ]:
def rjob(args):
    i, off = args
    s = S[i]
    areas = {a: s["X"][:, s["area"] == a] for a in ("LM", "PM") if (s["area"] == a).sum() >= 2}
    return K.random_measures(V1[i], s["labels"], {k: v + off for k, v in SEEDS.items()}, areas=areas)
t = time.time()
seed_res = {}
for off in (1000, 2000, 3000, 4000):
    with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
        seed_res[off] = dict(zip(keep, ex.map(rjob, [(i, off) for i in keep])))
print(f"seed checks in {(time.time() - t) / 60:.1f} min")

In [ ]:
rows = []
for off, R_ in seed_res.items():
    c4 = pd.DataFrame({i: R_[i]["rsm_half"] for i in R_}).T[SLOPES].mean()
    e4 = pd.DataFrame({p: {c: rsm_sp(R_[p[0]]["rsm"][c], R_[p[1]]["rsm"][c]) for c in SLOPES} for p in pairs}).T.mean()
    row = dict(offset=off, fig4c_peak=c4.idxmax(), fig4c_max=round(c4.max(), 4), fig4e_peak=e4.idxmax(), fig4e_max=round(e4.max(), 4))
    for a in ("LM", "PM"):
        ids = [i for i in R_ if a in R_[i]["area_rsm_half"]]
        wa = pd.DataFrame({i: R_[i]["area_rsm_half"][a] for i in ids}).T[SLOPES]
        pa = list(itertools.combinations(ids, 2))
        aa = pd.DataFrame({p: {c: rsm_sp(R_[p[0]]["area_rsm"][a][c], R_[p[1]]["area_rsm"][a][c]) for c in SLOPES} for p in pa}).T
        row[f"{a}_within_peak"], row[f"{a}_within_1_vs_max_minus_sem"] = wa.mean().idxmax(), bool(wa.mean()[1.0] >= wa.mean().max() - wa.sem()[wa.mean().idxmax()])
        row[f"{a}_across_peak"], row[f"{a}_across_1_vs_max_minus_sem"] = aa.mean().idxmax(), bool(aa.mean()[1.0] >= aa.mean().max() - aa.sem()[aa.mean().idxmax()])
    rows.append(row)
print(pd.DataFrame(rows).set_index("offset").to_string())
print(f"notebook time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: seed check.** Under the seed plus 1000, 2000, 3000 and 4000, the peak of fig4c (within sessions) and of fig4e (across sessions) is at slope 1.1 each time (maxima 0.439 to 0.449 and 0.388), as under the claims' seeds. For figs6, the mean at slope 1 lies within one SEM of the maximum in all four panels under every seed; the within-session peak moves between 0.3 and 1.0 (LM) and 0.3 and 0.5 (PM), the across-pair peak between 1.1 and 1.2 (LM) and 0.9 and 1.0 (PM).

## Summary

| Claim | Outcome |
|---|---|
| fig2b-decoding-falls-with-slope | blocked, too large (as-is and slopes 0 and 1 run in 30 sessions, slope 2 in four; slope 0 above slope 1 and as-is in every session, slope 2 below as-is in the four, all above chance) |
| figs4c-shuffling-alters-dependence | reproduced (decoding at as-is and slope 0, RSM similarity within and across sessions) |

How the analysis differs from the paper's: the decoding was run at three of 22 conditions over the 30 sessions and at a fourth, slope 2, in four sessions, under the claim's seed only; the shuffled comparisons were run under the claim's seed only. Units are those of the second notebook (the Methods' criteria and the label "good"). The random trial orders, folds and shuffles use this attempt's seeds, not the code's `np.random.seed(0)` sequence.

What was found about the data: nothing beyond the second notebook.